# 01 · Dataset audit, validation and augmentation QA
Exit criteria: all unit + data tests pass, split lock verified, license register reviewed, augmentation gallery approved by a human.

## 0. Session setup (run every session)
Code is cloned from GitHub (`moghbaie/InspectEdge_Mobile_VLM`); nothing needs to be uploaded.
Google Drive (`MyDrive/InspectEdge`) only stores outputs: reports, manifests, checkpoints.
Runtime → Change runtime type → **GPU** (T4 is enough for 500M; L4/A100 for 3B).

In [ ]:
import os, json, subprocess
from google.colab import drive
drive.mount('/content/drive')
REPO  = 'https://github.com/moghbaie/InspectEdge_Mobile_VLM.git'
DRIVE = '/content/drive/MyDrive/InspectEdge'   # persistent outputs: reports, manifests, checkpoints
WORK  = '/content/InspectEdge'                 # fast local disk: code + raw images
os.makedirs(DRIVE, exist_ok=True)
if os.path.isdir(f'{WORK}/.git'):
    !git -C {WORK} pull -q --ff-only
else:
    !git clone -q {REPO} {WORK}
assert os.path.exists(f'{WORK}/pyproject.toml'), 'clone failed: check REPO URL / repo visibility'
%cd {WORK}
!git log --oneline -1
!pip -q install -r requirements-colab.txt
!pip -q install -e . --no-deps
os.environ['INSPECTEDGE_ROOT'] = WORK
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, transformers, peft
assert transformers.__version__ == '4.57.1', 'pinned transformers not active: Runtime -> Restart session, then re-run this cell'
print('torch', torch.__version__, 'cuda', torch.version.cuda, '| transformers', transformers.__version__, '| peft', peft.__version__)
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'

In [ ]:
# Data: official VisA release (CC BY 4.0). Verified by SHA-256 before use.
VISA_SHA = '2eb8690c803ab37de0324772964100169ec8ba1fa3f7e94291c9ca673f40f362'
if not os.path.exists('data/raw/visa/pcb1'):
    os.makedirs('data/raw/_dl', exist_ok=True); os.makedirs('data/raw/visa', exist_ok=True)
    !wget -q -O data/raw/_dl/VisA_20220922.tar https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar
    h = subprocess.run(['sha256sum', 'data/raw/_dl/VisA_20220922.tar'], capture_output=True, text=True).stdout.split()[0]
    assert h == VISA_SHA, f'VisA archive checksum mismatch: {h}'
    !tar -xf data/raw/_dl/VisA_20220922.tar -C data/raw/visa && rm data/raw/_dl/VisA_20220922.tar
!ls data/raw/visa

In [ ]:
# Manifest: rebuilt deterministically each session; fails loudly if the locked split changed.
!python -m inspectedge.data.build_manifest

## 1. Unit + data validation tests

In [ ]:
!python -m pytest tests -q -rA 2>&1 | tail -25

## 2. License register (manual gate)

In [ ]:
import pandas as pd; pd.read_csv('data/licenses/data_sources.csv')[['source_id','license','commercial_training','status']]

## 3. Data sufficiency statistics

In [ ]:
!python scripts/dataset_stats.py

## 4. Augmentation visual QA (approve 50–100 examples per family before training)

In [ ]:
from IPython.display import Image, display
for c in ['pcb1', 'pcb4', 'capsules', 'candle']:
    !python scripts/aug_gallery.py {c}
    display(Image(f'reports/aug_gallery_{c}.jpg', width=700))

## 5. B0 image-quality gate

In [ ]:
!python scripts/b0_quality_gate.py

In [ ]:
# Persist outputs to Drive (run at the end and after long steps)
!rsync -aR reports data/manifests {DRIVE}/
print('synced to', DRIVE)